In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path
import sys
from typing import Optional, List, Dict, Any, Tuple
if '..' not in sys.path: sys.path.append('..')

from dataclasses import dataclass
from datasets import load_dataset
from datasets.arrow_dataset import Dataset
import numpy as np
from matplotlib import pyplot as plt
from pydantic_yaml import parse_yaml_file_as
import torch
from torch import nn
import torch.nn.functional as F
from transformers import AutoTokenizer, PreTrainedTokenizer

from mllm.data.wiki.dswiki import WikiDsLoader
from mllm.data.utils import RandomInputTokenizer, RandomInputTokenizerV2, TokensSubset, TokensSubsetV2, tokens_subsets_to_tensors, tokens_subsets_v2_to_tensors
from mllm.exp.args import MIXED_DECODER_MODEL_CFG_FNAME
from mllm.model.mixed_decoder import MixedDecoder
from mllm.config.model import MixedDecoderCfg
from mllm.train.encdec_graph_bert import MaskedCiteDataset, create_masked_cite_dataloader, load_split_wiki_dataset

In [ ]:
# DATA_PATH = Path(os.path.expandvars('$HOME')) / 'data'
DATA_PATH = Path('Q:/data')
# WIKI_DS_NAME = '20200501.en'
WIKI_DS_NAME = '20220301.en'

TRAIN_ENCDEC_BERT_PATH = DATA_PATH / 'train_mllm_encdec_bert'
mixed_decoder_subdir = 'mixeddecoder-20260304_105309-pre_encdecbert20260110193915-bertbaseuncased-d768-embEncCls-inp128-decGpt2-decmgpt2-msl384-sepT-pallF-eer4-ewn10x10-frzencF-trn_lr5e-05_bs30'

mixed_decoder_train_path = TRAIN_ENCDEC_BERT_PATH / mixed_decoder_subdir
mixed_decoder_snapshot_fpath = mixed_decoder_train_path / 'best.pth'
mixed_decoder_model_cfg_fpath = mixed_decoder_train_path / MIXED_DECODER_MODEL_CFG_FNAME

device_name = 'cpu'
# device_name = 'cuda'

device = torch.device(device_name)
print(device)

## Wikipedia dataset loading

In [ ]:
# dss = load_dataset('wikipedia', WIKI_DS_NAME, beam_runner='DirectRunner', cache_dir=str(DATA_PATH))
dss = load_dataset('wikipedia', WIKI_DS_NAME, cache_dir=str(DATA_PATH), trust_remote_code=True)
ds: Dataset = dss['train']
n_docs = len(ds)
print(f'Wikipedia {WIKI_DS_NAME} docs: {n_docs}')

### Masked dataset loading

In [ ]:
model_cfg = parse_yaml_file_as(MixedDecoderCfg, mixed_decoder_model_cfg_fpath)
print(model_cfg.dict())

tkz = AutoTokenizer.from_pretrained(model_cfg.enc_bert.pretrained_model_name)
inp_len = model_cfg.enc_bert.inp_len
n_special_toks = 3
prompt_all = model_cfg.prompt_all
mask_cfg = None
ds_cite = MaskedCiteDataset(ds, tkz, max_seq_len=inp_len, n_special_toks=n_special_toks, mask_cfg=mask_cfg, prompt_all=prompt_all, device=device)

### Inspect a masked dataset batch samples

In [ ]:
batch_size = 5
inds = np.arange(batch_size)
inds = inds.tolist()
batch = ds_cite.get_batch(inds)

In [ ]:
print(tkz)

In [ ]:
print(f'inp_toks: {batch.inp_toks.shape}. inp_attn_mask: {batch.inp_att_mask.shape}. first: {batch.inp_toks[0, 0]}. last: {batch.inp_toks[0, -1]}.')
print(f'prompts_toks: {batch.prompts_toks.shape}. prompts_attn_mask: {batch.prompts_att_mask.shape}. first: {batch.prompts_toks[0, 0]}. last: {batch.prompts_toks[0, -1]}.')
print(f'cites_toks: {batch.cites_toks.shape}. cites_attn_mask: {batch.cites_att_mask.shape}.')

In [ ]:
print('inp_toks[0]:', tkz.decode(batch.inp_toks[0].cpu().numpy()))
print('prompts_toks[0]:', tkz.decode(batch.prompts_toks[0].cpu().numpy()))
print('cites_toks[0]:', tkz.decode(batch.cites_toks[0].cpu().numpy()))

## Model loading and inference

In [ ]:
chkpt = torch.load(mixed_decoder_snapshot_fpath, map_location=device)
model = MixedDecoder(model_cfg, tkz).to(device)
model.load_pretrained(chkpt)
del chkpt
model.eval()
None

### Run forward pass (with windowing disabled for inference)

In [ ]:
# Temporarily disable windowing for deterministic inference
orig_emb_win_max_size = model.cfg.emb_win_max_size
model.cfg = model.cfg.copy(update={'emb_win_max_size': 0})

batch_off = 0
batch_size = 5
inds = np.arange(batch_size) + batch_off
inds = inds.tolist()
batch = ds_cite.get_batch(inds)

In [ ]:
with torch.no_grad():
    loss_dict, dec_logits = model.run_on_text_citation(batch)
print(loss_dict)

In [ ]:
logits = dec_logits.detach().numpy()
logits.shape

In [ ]:
probs_pred = torch.softmax(dec_logits, dim=-1)
print('probs shape:', probs_pred.shape)
docs_toks_out = torch.argmax(probs_pred, dim=-1)
docs_toks_out = docs_toks_out.detach().numpy()
print('toks_out shape:', docs_toks_out.shape)

In [ ]:
# Compute target_start_idx to extract only the target region from logits
# Layout: [ctx_embs(n_ctx), (sep), prompt_toks, target_toks_input(T-1)]
# Target labels start at target_start_idx = prefix_len - 1 (last prompt position predicts first target)
n_ctx = batch_size  # all batch CLS embeddings used as context
if model.cfg.emb_exp_rate > 0:
    n_ctx = batch_size * model.cfg.emb_exp_rate
prefix_len = n_ctx
if model.cfg.use_sep:
    prefix_len += 1
prefix_len += batch.prompts_toks.shape[1]
target_start_idx = prefix_len - 1

if model.cfg.prompt_all:
    target_len = batch.inp_toks.shape[1] - 1  # strip CLS
else:
    target_len = batch.cites_toks.shape[1]

print(f'n_ctx: {n_ctx}, prefix_len: {prefix_len}, target_start_idx: {target_start_idx}, target_len: {target_len}')

# Extract predicted tokens for the target region
target_logits = dec_logits[:, target_start_idx:target_start_idx + target_len, :]
target_toks_pred = torch.argmax(target_logits, dim=-1).detach().numpy()
print('target_toks_pred shape:', target_toks_pred.shape)

In [ ]:
for i in range(batch_size):
    print(f'=== Sample {i} ===')
    if model.cfg.prompt_all:
        gt_toks = batch.inp_toks[i, 1:]  # strip CLS
    else:
        gt_toks = batch.cites_toks[i]
    print('ground truth:')
    print(tkz.decode(gt_toks.cpu().numpy()))
    print('predicted:')
    print(tkz.decode(target_toks_pred[i]))
    print()

## Encoder embeddings inspection

In [ ]:
with torch.no_grad():
    inp_enc_embs = model.run_enc(batch.inp_toks, batch.inp_att_mask)
print('inp_enc_embs shape:', inp_enc_embs.shape)

inp_embs = inp_enc_embs.cpu().detach().numpy()
print('min:', inp_embs.min(), 'max:', inp_embs.max(), 'mean:', inp_embs.mean(), 'std:', inp_embs.std())

## Autoregressive generation from encoder embeddings

In [ ]:
@torch.no_grad()
def generate_from_embs(
    model: MixedDecoder, inp_enc_embs: torch.Tensor,
    prompt_toks: torch.Tensor, prompt_att_mask: torch.Tensor,
    max_new_tokens: int = 100, temperature: float = 1.0,
) -> torch.Tensor:
    """Autoregressive generation: given encoder CLS embeddings and prompt tokens,
    generate tokens one by one using the MixedDecoder.

    Args:
        model: MixedDecoder model in eval mode
        inp_enc_embs: (batch_size, d_model) - CLS embeddings from encoder
        prompt_toks: (batch_size, prompt_len) - prompt token ids
        prompt_att_mask: (batch_size, prompt_len) - prompt attention mask
        max_new_tokens: maximum number of tokens to generate
        temperature: sampling temperature (1.0 = greedy with argmax)

    Returns:
        generated_toks: (batch_size, max_new_tokens) - generated token ids
    """
    batch_size = inp_enc_embs.shape[0]
    device = inp_enc_embs.device
    cfg = model.cfg

    # Build context embeddings (no windowing for generation)
    if cfg.emb_exp_rate > 0:
        exp_embs = model.emb_exp(inp_enc_embs)
        exp_embs = exp_embs.view(batch_size, cfg.emb_exp_rate, model.d_dec)
        exp_embs = exp_embs.reshape(1, batch_size * cfg.emb_exp_rate, model.d_dec)
        ctx_embs = exp_embs.expand(batch_size, -1, -1)
    else:
        ctx_embs = inp_enc_embs.unsqueeze(0).expand(batch_size, -1, -1)

    # Project if needed
    if model.enc_proj is not None:
        ctx_embs = model.enc_proj(ctx_embs)

    n_ctx = ctx_embs.shape[1]

    # Build initial prefix: [ctx_embs, (sep), prompt_embs]
    parts_embs = [ctx_embs]
    parts_mask = [torch.ones((batch_size, n_ctx), dtype=torch.long, device=device)]

    if cfg.use_sep:
        sep_tok = torch.full((batch_size, 1), model.sep_token_id, dtype=torch.long, device=device)
        sep_emb = model.word_embeddings(sep_tok)
        parts_embs.append(sep_emb)
        parts_mask.append(torch.ones((batch_size, 1), dtype=torch.long, device=device))

    prompt_embs = model.word_embeddings(prompt_toks)
    parts_embs.append(prompt_embs)
    parts_mask.append(prompt_att_mask)

    input_embs = torch.cat(parts_embs, dim=1)  # (batch_size, prefix_len, d_dec)
    attention_mask = torch.cat(parts_mask, dim=1)

    generated_ids = []
    for step in range(max_new_tokens):
        total_len = input_embs.shape[1]
        pos_ids = torch.arange(total_len, device=device).unsqueeze(0)
        pos_embs = model.pos_emb(pos_ids)
        embs_with_pos = input_embs + pos_embs

        logits = model.run_decoder(embs_with_pos, attention_mask)
        next_logits = logits[:, -1, :]  # (batch_size, vocab_size)

        if temperature <= 0:
            next_tok = torch.argmax(next_logits, dim=-1)  # (batch_size,)
        else:
            probs = torch.softmax(next_logits / temperature, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).squeeze(-1)

        generated_ids.append(next_tok)

        # Append new token embedding
        next_emb = model.word_embeddings(next_tok.unsqueeze(1))  # (batch_size, 1, d_dec)
        input_embs = torch.cat([input_embs, next_emb], dim=1)
        attention_mask = torch.cat([attention_mask, torch.ones((batch_size, 1), dtype=torch.long, device=device)], dim=1)

    return torch.stack(generated_ids, dim=1)  # (batch_size, max_new_tokens)

In [ ]:
max_new_tokens = 100
gen_toks = generate_from_embs(
    model, inp_enc_embs,
    batch.prompts_toks, batch.prompts_att_mask,
    max_new_tokens=max_new_tokens, temperature=0,
)
print('gen_toks shape:', gen_toks.shape)

for i in range(batch_size):
    print(f'=== Sample {i} ===')
    print('input:')
    print(tkz.decode(batch.inp_toks[i].cpu().numpy()))
    print('prompt:')
    print(tkz.decode(batch.prompts_toks[i].cpu().numpy()))
    print('generated:')
    print(tkz.decode(gen_toks[i].cpu().numpy()))
    print()

## Decoding with noisy embeddings

In [ ]:
# Add small noise to encoder embeddings and compare generation quality
noise_std = 1e-2
noise = torch.randn_like(inp_enc_embs) * noise_std
inp_enc_embs_noisy = inp_enc_embs + noise

gen_toks_noisy = generate_from_embs(
    model, inp_enc_embs_noisy,
    batch.prompts_toks, batch.prompts_att_mask,
    max_new_tokens=max_new_tokens, temperature=0,
)

for i in range(batch_size):
    print(f'=== Sample {i} ===')
    print('input:')
    print(tkz.decode(batch.inp_toks[i].cpu().numpy()))
    print('generated (clean):')
    print(tkz.decode(gen_toks[i].cpu().numpy()))
    print('generated (noisy, std={:.0e}):'.format(noise_std))
    print(tkz.decode(gen_toks_noisy[i].cpu().numpy()))
    print()

## Restore original config

In [ ]:
# Restore original emb_win_max_size
model.cfg = model.cfg.copy(update={'emb_win_max_size': orig_emb_win_max_size})
print('emb_win_max_size restored to:', model.cfg.emb_win_max_size)